In [1]:
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

c:\Users\falak\Desktop\Seryn_revamp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ------------------------------------------------
# 0) Inputs
# ------------------------------------------------
FOOD_FILE = "food_labels_with_gi_label.csv"   # Dish Name, Meal Type, Glycemic Index, GI_Label
PATIENT_PROFILE = {
    "Age": 58,
    "Sex": "female",
    "DiabetesType": "Type 2",
    "HbA1c": 7.3,
    "CalorieNeeds": 1660,
    "CarbTolerance": 131,
    "SugarSensitivity": "High",
    "ActivityLevel": "Moderate",
}

In [3]:
# ------------------------------------------------
# 1) Safe foods pool based on GI_Label
# ------------------------------------------------
df = pd.read_csv(FOOD_FILE)

# keep only friendly + limited
safe_pool = df[df["GI_Label"].str.lower().isin(["friendly", "limited"])]

# Helper
def pool_by_meal(df, meal, k=40):
    x = df[df["Meal Type"].str.lower() == meal.lower()]
    return sorted(
        x["Dish Name"].dropna().astype(str).str.strip().str.lower().unique().tolist()
    )[:k]

safe_breakfast = pool_by_meal(safe_pool, "Breakfast", k=40)
safe_lunch     = pool_by_meal(safe_pool, "Lunch", k=40)
safe_dinner    = pool_by_meal(safe_pool, "Dinner", k=40)
safe_snacks    = pool_by_meal(safe_pool, "Snacks", k=30)
safe_drinks    = pool_by_meal(safe_pool, "Drinks", k=30)

In [4]:
# ------------------------------------------------
# 2) Prompt template (forces JSON output)
# ------------------------------------------------
def plan_schema():
    return {f"Day{i}": {"Breakfast":"","Lunch":"","Dinner":"","Snack":"","Drink":""} for i in range(1,8)}

def make_prompt(profile, bfoods, lfoods, dfoods, sfoods, drfoods):
    schema = json.dumps(plan_schema(), indent=2)
    return f"""
You are a meticulous dietitian. Create a 7-day culturally appropriate Emirati diabetic-friendly meal plan.

PATIENT:
- Age: {profile['Age']}
- Sex: {profile['Sex']}
- Diabetes Type: {profile['DiabetesType']}
- HbA1c: {profile['HbA1c']}
- Daily Calorie Target: {profile['CalorieNeeds']} kcal
- Carb Tolerance: {profile['CarbTolerance']} g/day
- Sugar Sensitivity: {profile['SugarSensitivity']}
- Activity Level: {profile['ActivityLevel']}

CONSTRAINTS:
- Stay close to {profile['CalorieNeeds']} kcal/day and {profile['CarbTolerance']} g carbs/day.
- Use ONLY items from the allowed shortlists below.
- Prefer 'friendly' foods; 'limited' may appear in small portions.
- Avoid added sugars; prefer low-GI carbs and high-fiber sides.
- Portion control is mandatory. Include short portion notes.

ALLOWED SHORTLISTS (use items from these lists only):
- Breakfast: {bfoods}
- Lunch: {lfoods}
- Dinner: {dfoods}
- Snacks: {sfoods}
- Drinks: {drfoods}

OUTPUT:
Return STRICT JSON ONLY following this schema (no commentary). Keep dishes as strings with brief portion notes:

{schema}
""".strip()

prompt = make_prompt(PATIENT_PROFILE, safe_breakfast, safe_lunch, safe_dinner, safe_snacks, safe_drinks)

In [5]:
# ------------------------------------------------
# 3) Model
# ------------------------------------------------
MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype="auto",
    device_map="auto"
)

gen = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Conservative decoding
outputs = gen(
    prompt,
    max_new_tokens=1200,
    do_sample=False,
    temperature=0.0,
    top_p=1.0,
    repetition_penalty=1.05,
)

raw = outputs[0]["generated_text"]

`torch_dtype` is deprecated! Use `dtype` instead!
Some parameters are on the meta device because they were offloaded to the disk and cpu.
Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [6]:
# ------------------------------------------------
# 4) Parse to JSON
# ------------------------------------------------
def extract_json(s: str):
    s = s.strip()
    start, end = s.find("{"), s.rfind("}")
    if start != -1 and end != -1 and end > start:
        s = s[start:end+1]
    return json.loads(s)

try:
    plan = extract_json(raw)
except Exception:
    plan = plan_schema()

# Save
with open("weekly_plan.json", "w", encoding="utf-8") as f:
    json.dump(plan, f, ensure_ascii=False, indent=2)

print("✅ Saved weekly_plan.json")


✅ Saved weekly_plan.json
